# Notebook 13 — Improving Weibull and RSF

**Goal:** improve our predictions of **years until a bridge first leaves Good condition (reaches Fair or Poor)**.

We test different versions of Weibull AFT and Random Survival Forest (RSF) to see whether they predict deterioration better.

**How we test improvements:** train on older bridge data, then compare predictions using bridges still Good in 2016 and followed during 2017–2020. We do **not** use the final 2021–2025 test set to choose the best settings.

**What we change:**
- **Both models:** add bridge condition ratings and state information.
- **Weibull:** try different ways to represent age and traffic, adjust regularisation, and allow a more flexible deterioration curve.
- **RSF:** change the number of trees, tree depth and minimum leaf size.

A higher validation C-index suggests better ranking of bridges by deterioration timing. **Notebook 11 will handle the final, consistent model comparison.**

In [1]:
from pathlib import Path
import gc, time, warnings
import numpy as np
import pandas as pd
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

warnings.filterwarnings("ignore", category=FutureWarning)

# Project folders (same logic as the other notebooks).
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent
FINAL = PROJECT / "data" / "final"
PROCESSED = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# ---- Settings you can change ----
CUTOFF = 2016                 # historical validation starts after this year
VAL_YEARS = range(2017, 2021) # validation follow-up years
SAMPLE_N = 150_000            # training bridges used in experiments (None = all). Smaller = faster.
RSF_TREES_BASE = 50           # trees for feature comparisons (matches Notebook 07)
RSF_TREES_GRID = 100          # trees for the settings grid
N_JOBS = 2                    # forest threads (raise if you have memory)
RARE_MIN = 200                # categories rarer than this are grouped as "other"
N_BOOT = 30                   # bootstrap resamples for the final CI (0 = skip)
RANDOM_STATE = 42
RUN_FINAL_TEST = False        # set True ONCE at the end, after choosing the best variants
print("Project:", PROJECT)

Project: /Users/annie/Downloads/bridge-deterioration


## 1. Prepare historical training and validation data

Use earlier years to train the models and 2017–2020 to check whether changes help.

In [2]:
train = pd.read_parquet(FINAL / "bridge_survival_train.parquet")
print("Training bridges:", f"{len(train):,}")

BASE_NUM = ["bridge_age", "ADT_029", "STRUCTURE_LEN_MT_049"]
BASE_CAT = ["STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B"]

# Historical training: first-Good episodes that started before the cutoff, outcomes censored at the cutoff.
hist = train.loc[train["start_year"] < CUTOFF].copy()
hist["h_event"] = ((hist["event"] == 1) & (hist["end_year"] <= CUTOFF)).astype(int)
hist["h_end"] = hist["end_year"].clip(upper=CUTOFF)
hist["h_duration"] = hist["h_end"] - hist["start_year"]
hist = hist.loc[hist["h_duration"] > 0].copy()

print("Historical training bridges:", f"{len(hist):,}")
print("Historical events (left Good by cutoff):", f"{hist['h_event'].sum():,}")

Training bridges: 498,875
Historical training bridges: 475,720
Historical events (left Good by cutoff): 207,007


In [3]:
# Bridges that are Good at the cutoff and whose first Good episode is still running.
snap_cut = pd.read_parquet(PROCESSED / f"nbi_{CUTOFF}_clean.parquet", columns=["bridge_id", "condition"])
good_cut = set(snap_cut.loc[snap_cut["condition"] == "Good", "bridge_id"])

pool = hist.loc[
    (hist["h_event"] == 0) & (hist["h_end"] == CUTOFF) & hist["bridge_id"].isin(good_cut)
].copy()
pool_ids = set(pool["bridge_id"])

# Observe them year by year after the cutoff.
frames = []
for yr in VAL_YEARS:
    s = pd.read_parquet(PROCESSED / f"nbi_{yr}_clean.parquet", columns=["bridge_id", "condition"])
    s = s[s["bridge_id"].isin(pool_ids)].drop_duplicates("bridge_id")
    s["observation_year"] = yr
    frames.append(s)
obs = pd.concat(frames, ignore_index=True)

wide = obs.pivot(index="bridge_id", columns="observation_year", values="condition").reindex(
    index=pool["bridge_id"].to_numpy(), columns=list(VAL_YEARS)
)

# Event = FIRST year the bridge is Fair or Poor (i.e. it leaves Good).
event = np.zeros(len(wide), dtype=int)
dur = np.zeros(len(wide), dtype=int)
active = np.ones(len(wide), dtype=bool)
for yr in VAL_YEARS:
    cond = wide[yr]
    left_good = active & cond.isin(["Fair", "Poor"]).to_numpy()
    still_good = active & cond.eq("Good").to_numpy()
    event[left_good] = 1
    dur[left_good] = yr - CUTOFF
    dur[still_good] = yr - CUTOFF
    active = still_good          # stop following after the event, a gap, or an Unknown rating

outcomes = pd.DataFrame({"bridge_id": wide.index, "event": event, "test_duration": dur})
val = outcomes.merge(pool.drop(columns=["event", "duration"]), on="bridge_id", validate="one_to_one")
val = val.loc[val["test_duration"] > 0].copy()
val["entry_duration"] = CUTOFF - val["start_year"]

print("Validation bridges:", f"{len(val):,}")
print("Validation events:", f"{val['event'].sum():,}", f"({val['event'].mean():.1%})")
print("Entry duration:", val["entry_duration"].min(), "to", val["entry_duration"].max())

Validation bridges: 178,551
Validation events: 29,369 (16.4%)
Entry duration: 1 to 24


## 2. Add bridge information

Test whether knowing the bridge's state and condition ratings helps predict when it will leave Good condition. These ratings come from its **first observed Good year**, not necessarily 2016.

In [4]:
EXTRA_SRC = ["bridge_id", "STATE_CODE_001", "DECK_COND_058", "SUPERSTRUCTURE_COND_059",
             "SUBSTRUCTURE_COND_060", "CULVERT_COND_062", "condition_rating"]
RATING_COLS = ["deck_rating", "super_rating", "sub_rating", "culvert_rating", "min_rating"]

def add_baseline_extras(df):
    # Attach state and ratings recorded in each bridge's first Good year (its start_year).
    parts = []
    for yr, grp in df.groupby("start_year"):
        snap = pd.read_parquet(PROCESSED / f"nbi_{int(yr)}_clean.parquet", columns=EXTRA_SRC)
        snap = snap.drop_duplicates("bridge_id")
        parts.append(snap[snap["bridge_id"].isin(grp["bridge_id"])])
    extras = pd.concat(parts, ignore_index=True).rename(columns={
        "STATE_CODE_001": "state", "DECK_COND_058": "deck_rating",
        "SUPERSTRUCTURE_COND_059": "super_rating", "SUBSTRUCTURE_COND_060": "sub_rating",
        "CULVERT_COND_062": "culvert_rating", "condition_rating": "min_rating"})
    out = df.merge(extras, on="bridge_id", how="left", validate="one_to_one")
    out["state"] = out["state"].astype("string")
    for c in RATING_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

hist = add_baseline_extras(hist)
val = add_baseline_extras(val)

print("Missing state (train / val):", int(hist["state"].isna().sum()), "/", int(val["state"].isna().sum()))
print("\nRatings in the first Good year (training):")
print(hist[RATING_COLS].describe().round(2).T[["count", "mean", "min", "max"]])
print("\nOverall rating distribution:")
print(hist["min_rating"].value_counts(dropna=False).sort_index())
print("\nBridges per state (top 8):")
print(hist["state"].value_counts().head(8))

Missing state (train / val): 0 / 0

Ratings in the first Good year (training):
                   count  mean  min  max
deck_rating     347229.0  7.79  7.0  9.0
super_rating    348961.0  7.87  7.0  9.0
sub_rating      349448.0  7.81  7.0  9.0
culvert_rating  126471.0  7.65  0.0  9.0
min_rating      475720.0  7.64  7.0  9.0

Overall rating distribution:
min_rating
7.0    240273
8.0    166055
9.0     69392
Name: count, dtype: int64

Bridges per state (top 8):
state
48    45892
17    34753
29    28104
06    18849
39    18823
19    18646
21    18075
20    18007
Name: count, dtype: Int64


In [5]:
# Training sample for the experiments (validation bridges are never sampled).
if SAMPLE_N is not None and SAMPLE_N < len(hist):
    train_s = hist.sample(SAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    train_s = hist.reset_index(drop=True)
val = val.reset_index(drop=True)
print("Experiment training bridges:", f"{len(train_s):,}", "| events:", f"{train_s['h_event'].sum():,}")
print("Validation bridges:", f"{len(val):,}", "| events:", f"{val['event'].sum():,}")

Experiment training bridges: 150,000 | events: 65,167
Validation bridges: 178,551 | events: 29,369


## 3. Prepare model inputs and predictions

Convert the bridge information into the numerical format each model needs. The helper functions also calculate predictions and validation scores.

In [6]:
def make_design(tr, other, num, cat=(), rating_cols=(), log_cols=(), age_sq=False, scale=False):
    # Build model inputs from `tr`, applying exactly the same fitted choices to `other`.
    X_tr = pd.DataFrame(index=tr.index)
    X_ot = pd.DataFrame(index=other.index)
    cont = []
    for c in list(num) + list(rating_cols):
        t = pd.to_numeric(tr[c], errors="coerce")
        o = pd.to_numeric(other[c], errors="coerce")
        if c in rating_cols:                         # remember where a rating was not recorded
            X_tr[c + "_na"] = t.isna().astype(int)
            X_ot[c + "_na"] = o.isna().astype(int)
        med = t.median()
        med = 0.0 if not np.isfinite(med) else med
        t, o = t.fillna(med), o.fillna(med)
        if age_sq and c == "bridge_age":
            X_tr["bridge_age_sq"], X_ot["bridge_age_sq"] = t ** 2, o ** 2
            cont.append("bridge_age_sq")
        if c in log_cols:
            t, o = np.log1p(t.clip(lower=0)), np.log1p(o.clip(lower=0))
        X_tr[c], X_ot[c] = t, o
        cont.append(c)
    if scale:                                        # standardise using training values only
        for c in cont:
            mu, sd = X_tr[c].mean(), X_tr[c].std()
            if sd and np.isfinite(sd):
                X_tr[c] = (X_tr[c] - mu) / sd
                X_ot[c] = (X_ot[c] - mu) / sd
    for c in cat:
        t = tr[c].astype("string").fillna("__missing__")
        o = other[c].astype("string").fillna("__missing__")
        counts = t.value_counts()
        keep = set(counts[counts >= RARE_MIN].index)
        t = t.where(t.isin(keep), "__other__")
        o = o.where(o.isin(keep), "__other__")
        cats = sorted(set(t.astype(str)) | {"__other__", "__missing__"})
        for series, X in ((t, X_tr), (o, X_ot)):
            d = pd.get_dummies(
                pd.Series(pd.Categorical(series.astype(str), categories=cats), index=series.index),
                prefix=c, drop_first=True, dtype=int)
            for col in d.columns:
                X[col] = d[col]
    X_ot = X_ot[X_tr.columns]
    keep_cols = [c for c in X_tr.columns if X_tr[c].nunique() > 1]   # drop constant columns
    return X_tr[keep_cols].astype(float), X_ot[keep_cols].astype(float)

FEATURE_SETS = {
    "base":                dict(num=BASE_NUM, cat=BASE_CAT),
    "base+ratings":        dict(num=BASE_NUM, cat=BASE_CAT, rating_cols=RATING_COLS),
    "base+state":          dict(num=BASE_NUM, cat=BASE_CAT + ["state"]),
    "base+ratings+state":  dict(num=BASE_NUM, cat=BASE_CAT + ["state"], rating_cols=RATING_COLS),
}
LOGS = ["ADT_029", "STRUCTURE_LEN_MT_049"]

Xa, Xb = make_design(train_s, val, scale=True, **FEATURE_SETS["base+ratings+state"])
print("Largest design matrix:", Xa.shape, "(columns:", Xa.shape[1], ")")
del Xa, Xb

Largest design matrix: (150000, 82) (columns: 82 )


In [7]:
def weibull_conditional_median(model, X, entry, ancillary=None):
    # Median remaining years given a bridge has already stayed Good for `entry` years.
    # S(t) = exp(-(t/lam)^rho): lam = time at S = e^-1, t50 = time at S = 0.5, so rho = ln(ln2)/ln(t50/lam).
    kw = {} if ancillary is None else {"ancillary": ancillary}
    lam = model.predict_percentile(X, p=np.exp(-1), **kw).to_numpy(float)
    t50 = model.predict_median(X, **kw).to_numpy(float)
    rho = np.log(np.log(2.0)) / np.log(t50 / lam)
    with np.errstate(over="ignore", invalid="ignore"):
        total = lam * np.exp(np.logaddexp(rho * np.log(np.maximum(entry, 1e-300) / lam), np.log(np.log(2.0))) / rho)
    return np.maximum(total - entry, 0.0), lam

def fit_weibull(train_df, eval_df, fs, penalizer=0.01, log_cols=(), age_sq=False, ancillary_cols=None):
    t0 = time.time()
    X_tr, X_ev = make_design(train_df, eval_df, scale=True, log_cols=log_cols, age_sq=age_sq, **fs)
    fit_df = X_tr.assign(duration=train_df["h_duration"].to_numpy(float), event=train_df["h_event"].to_numpy(int))
    model = WeibullAFTFitter(penalizer=penalizer)
    anc_tr = anc_ev = None
    if ancillary_cols:
        anc_tr, anc_ev = X_tr[ancillary_cols], X_ev[ancillary_cols]
        model.fit(fit_df, duration_col="duration", event_col="event", ancillary=anc_tr)
    else:
        model.fit(fit_df, duration_col="duration", event_col="event")
    remaining, lam = weibull_conditional_median(model, X_ev, eval_df["entry_duration"].to_numpy(float), anc_ev)
    c_med = concordance_index(eval_df["test_duration"], remaining, eval_df["event"])
    c_scale = concordance_index(eval_df["test_duration"], lam, eval_df["event"])
    return dict(n_features=X_tr.shape[1], train_c=model.concordance_index_, c_index=c_med,
                c_index_scale=c_scale, seconds=time.time() - t0), remaining

def fit_rsf(train_df, eval_df, fs, n_estimators=50, max_depth=12, min_samples_leaf=50,
            min_samples_split=100, max_features="sqrt", max_samples=0.5):
    t0 = time.time()
    X_tr, X_ev = make_design(train_df, eval_df, scale=False, **fs)
    y = Surv.from_arrays(event=train_df["h_event"].to_numpy(bool), time=train_df["h_duration"].to_numpy(float))
    rsf = RandomSurvivalForest(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                               min_samples_split=min_samples_split, max_features=max_features,
                               max_samples=max_samples, n_jobs=N_JOBS, random_state=RANDOM_STATE)
    rsf.fit(X_tr.to_numpy(np.float32), y)
    risk = rsf.predict(X_ev.to_numpy(np.float32))
    score = -risk                                   # larger = survives longer
    c = concordance_index(eval_df["test_duration"], score, eval_df["event"])
    out = dict(n_features=X_tr.shape[1], train_c=np.nan, c_index=c, c_index_scale=np.nan, seconds=time.time() - t0)
    del rsf; gc.collect()
    return out, score

ROWS, SCORES, CONFIGS = [], {}, {}
def record(model, name, stage, res, score, cfg):
    ROWS.append(dict(model=model, variant=name, stage=stage, **res))
    SCORES[(model, name)] = score
    CONFIGS[(model, name)] = cfg
    print(f"{model:12s} {name:34s} features={res['n_features']:3d}  validation C-index={res['c_index']:.4f}  ({res['seconds']:.0f}s)")

## 4. Improve Weibull AFT

First, test extra bridge information and different ways of representing age, traffic and length. Then try regularisation and a more flexible curve shape. Compare each version with the original Weibull model.

In [8]:
# Stage A: feature sets, with and without log-transformed traffic/length and a curved age term.
W_GRID = [
    ("W0 base (current)",             "base",               dict()),
    ("W1 base + log + age^2",         "base",               dict(log_cols=LOGS, age_sq=True)),
    ("W2 + ratings",                  "base+ratings",       dict(log_cols=LOGS, age_sq=True)),
    ("W3 + state",                    "base+state",         dict(log_cols=LOGS, age_sq=True)),
    ("W4 + ratings + state",          "base+ratings+state", dict(log_cols=LOGS, age_sq=True)),
]
for name, fs_key, kw in W_GRID:
    res, score = fit_weibull(train_s, val, FEATURE_SETS[fs_key], penalizer=0.01, **kw)
    record("Weibull AFT", name, "features", res, score, dict(fs=fs_key, penalizer=0.01, **kw))

Weibull AFT  W0 base (current)                  features= 24  validation C-index=0.6156  (4s)
Weibull AFT  W1 base + log + age^2              features= 25  validation C-index=0.6259  (4s)
Weibull AFT  W2 + ratings                       features= 34  validation C-index=0.6427  (5s)
Weibull AFT  W3 + state                         features= 74  validation C-index=0.6297  (7s)
Weibull AFT  W4 + ratings + state               features= 83  validation C-index=0.6427  (8s)


In [9]:
# Stage B: on the best feature set, try stronger regularisation and a shape that depends on age.
w_feat = [r for r in ROWS if r["model"] == "Weibull AFT" and r["stage"] == "features"]
best_w_name = max(w_feat, key=lambda r: r["c_index"])["variant"]
best_w_cfg = CONFIGS[("Weibull AFT", best_w_name)]
print("Best Weibull feature set so far:", best_w_name)

for pen in [0.1]:
    cfg = {**best_w_cfg, "penalizer": pen}
    fs_key = cfg.pop("fs")
    res, score = fit_weibull(train_s, val, FEATURE_SETS[fs_key], **cfg)
    record("Weibull AFT", f"{best_w_name} | penalizer {pen}", "tuning", res, score, {"fs": fs_key, **cfg})

cfg = {**best_w_cfg, "ancillary_cols": ["bridge_age"]}
fs_key = cfg.pop("fs")
res, score = fit_weibull(train_s, val, FEATURE_SETS[fs_key], **cfg)
record("Weibull AFT", f"{best_w_name} | shape depends on age", "tuning", res, score, {"fs": fs_key, **cfg})

Best Weibull feature set so far: W4 + ratings + state
Weibull AFT  W4 + ratings + state | penalizer 0.1 features= 83  validation C-index=0.6411  (7s)
Weibull AFT  W4 + ratings + state | shape depends on age features= 83  validation C-index=0.6444  (8s)


## 5. Improve Random Survival Forest

First, test extra bridge information. Then try different numbers of trees, tree depths and minimum leaf sizes. Compare each version with the original RSF.

In [10]:
# Stage A: feature sets with the current forest settings (same as Notebook 07).
R_GRID = [
    ("R0 base (current)",        "base"),
    ("R1 + ratings",             "base+ratings"),
    ("R2 + state",               "base+state"),
    ("R3 + ratings + state",     "base+ratings+state"),
]
for name, fs_key in R_GRID:
    res, score = fit_rsf(train_s, val, FEATURE_SETS[fs_key], n_estimators=RSF_TREES_BASE)
    record("RSF", name, "features", res, score,
           dict(fs=fs_key, n_estimators=RSF_TREES_BASE, max_depth=12, min_samples_leaf=50))

RSF          R0 base (current)                  features= 24  validation C-index=0.6275  (48s)
RSF          R1 + ratings                       features= 33  validation C-index=0.6391  (35s)
RSF          R2 + state                         features= 73  validation C-index=0.6291  (40s)
RSF          R3 + ratings + state               features= 82  validation C-index=0.6415  (35s)


In [11]:
# Stage B: on the best feature set, try more trees and different depth / leaf sizes.
r_feat = [r for r in ROWS if r["model"] == "RSF" and r["stage"] == "features"]
best_r_name = max(r_feat, key=lambda r: r["c_index"])["variant"]
best_r_fs = CONFIGS[("RSF", best_r_name)]["fs"]
print("Best RSF feature set so far:", best_r_name)

for depth, leaf in [(12, 50), (16, 25), (20, 25), (16, 50)]:
    res, score = fit_rsf(train_s, val, FEATURE_SETS[best_r_fs], n_estimators=RSF_TREES_GRID,
                         max_depth=depth, min_samples_leaf=leaf)
    record("RSF", f"{best_r_name} | {RSF_TREES_GRID} trees, depth {depth}, leaf {leaf}", "tuning", res, score,
           dict(fs=best_r_fs, n_estimators=RSF_TREES_GRID, max_depth=depth, min_samples_leaf=leaf))

Best RSF feature set so far: R3 + ratings + state
RSF          R3 + ratings + state | 100 trees, depth 12, leaf 50 features= 82  validation C-index=0.6418  (78s)
RSF          R3 + ratings + state | 100 trees, depth 16, leaf 25 features= 82  validation C-index=0.6440  (87s)
RSF          R3 + ratings + state | 100 trees, depth 20, leaf 25 features= 82  validation C-index=0.6451  (83s)
RSF          R3 + ratings + state | 100 trees, depth 16, leaf 50 features= 82  validation C-index=0.6439  (84s)


## 6. Compare the results

Look at the validation C-index for the original and improved versions. The bootstrap check gives an indication of how stable the observed improvement is, but it does not replace final testing.

In [12]:
results = pd.DataFrame(ROWS)
for m in results["model"].unique():
    base = results.loc[(results["model"] == m)].iloc[0]["c_index"]
    results.loc[results["model"] == m, "gain_vs_current"] = results.loc[results["model"] == m, "c_index"] - base
results["meaningful_gain"] = results["gain_vs_current"] > 0.005
results.to_csv(RESULTS / "rsf_weibull_improvement_experiments.csv", index=False)
show = results[["model", "variant", "stage", "n_features", "train_c", "c_index", "gain_vs_current", "meaningful_gain", "seconds"]]
print(show.round(4).to_string(index=False))
print("\nReminder: gains under about 0.005 are within noise; the first row of each model is the current setup.")

      model                                             variant    stage  n_features  train_c  c_index  gain_vs_current  meaningful_gain  seconds
Weibull AFT                                   W0 base (current) features          24   0.6768   0.6156           0.0000            False   4.0436
Weibull AFT                               W1 base + log + age^2 features          25   0.6887   0.6259           0.0103             True   3.9705
Weibull AFT                                        W2 + ratings features          34   0.7140   0.6427           0.0271             True   5.2423
Weibull AFT                                          W3 + state features          74   0.7067   0.6297           0.0141             True   7.2211
Weibull AFT                                W4 + ratings + state features          83   0.7277   0.6427           0.0271             True   8.4024
Weibull AFT                W4 + ratings + state | penalizer 0.1   tuning          83   0.7254   0.6411           0.0255     

In [13]:
# Uncertainty: how big is the difference between the current and the best variant of each model?
def paired_bootstrap(score_a, score_b, n=N_BOOT, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    dur_, ev_ = val["test_duration"].to_numpy(), val["event"].to_numpy()
    diffs = []
    for _ in range(n):
        idx = rng.integers(0, len(val), len(val))
        diffs.append(concordance_index(dur_[idx], score_b[idx], ev_[idx]) - concordance_index(dur_[idx], score_a[idx], ev_[idx]))
    return float(np.mean(diffs)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

BEST = {}
for m in ["Weibull AFT", "RSF"]:
    sub = results[results["model"] == m]
    best_row = sub.loc[sub["c_index"].idxmax()]
    BEST[m] = best_row["variant"]
    print(f"{m}: current {sub.iloc[0]['c_index']:.4f} -> best '{best_row['variant']}' {best_row['c_index']:.4f}")
    if N_BOOT and best_row["variant"] != sub.iloc[0]["variant"]:
        mean_d, lo, hi = paired_bootstrap(SCORES[(m, sub.iloc[0]["variant"])], SCORES[(m, best_row["variant"])])
        print(f"   paired bootstrap difference: {mean_d:+.4f}  (95% interval {lo:+.4f} to {hi:+.4f})")

Weibull AFT: current 0.6156 -> best 'W4 + ratings + state | shape depends on age' 0.6444
   paired bootstrap difference: +0.0285  (95% interval +0.0262 to +0.0316)
RSF: current 0.6275 -> best 'R3 + ratings + state | 100 trees, depth 20, leaf 25' 0.6451
   paired bootstrap difference: +0.0175  (95% interval +0.0157 to +0.0195)


## 7. Optional final-test check

This section is **off by default** (`RUN_FINAL_TEST = False`). It is intended to refit selected versions on all training bridges and check the final test data **once**, after model selection.

**Do not switch it on yet:** the final-test code still needs to be checked for column compatibility. Full-data training may also take considerable time and memory.

In [14]:
if RUN_FINAL_TEST:
    test = pd.read_parquet(FINAL / "bridge_survival_test_aligned.parquet")
    test = add_baseline_extras(test)
    full = add_baseline_extras(train)
    full["h_duration"] = full["duration"]
    full["h_event"] = full["event"]
    final_rows = []
    for m, fitter in [("Weibull AFT", fit_weibull), ("RSF", fit_rsf)]:
        names = [results[results["model"] == m].iloc[0]["variant"], BEST[m]]
        for nm in dict.fromkeys(names):                      # current first, then best (once if the same)
            cfg = dict(CONFIGS[(m, nm)])
            fs = FEATURE_SETS[cfg.pop("fs")]
            if m == "RSF":
                cfg["n_estimators"] = max(cfg.get("n_estimators", 50), 50)
            res, _ = fitter(full, test, fs, **cfg)
            final_rows.append(dict(model=m, variant=nm, full_test_c_index=res["c_index"], seconds=res["seconds"]))
            print(m, "|", nm, "| full-test C-index:", round(res["c_index"], 4))
    pd.DataFrame(final_rows).to_csv(RESULTS / "rsf_weibull_final_test_check.csv", index=False)
else:
    print("RUN_FINAL_TEST is False - the 2021 test set has not been touched.")

RUN_FINAL_TEST is False - the 2021 test set has not been touched.


## 8. Conclusion

We keep the versions that perform better on historical validation and compare them properly in Notebook 11.

**Limitations:** these scores are preliminary. They do not fully handle different times already spent in Good condition, and some older bridges were already Good when the dataset began. A higher C-index also does not prove that predicted years are accurate.